# 19 — Benchmark selection with a model-based topic filter

The earlier benchmark (notebook 00) assigned each MedQA question to one of the ten
target primary-care conditions by keyword matching. That approach both admits
questions that merely mention a condition in passing and misses questions that
never spell the keyword out. Here the same selection is driven by a language
model that judges the single condition a question is mainly about, and a balanced
sample of 200 is drawn with an equal share per condition. The output columns match
the format the retrieval notebook consumes.

In [12]:
# Environment and Drive
from google.colab import drive, userdata
import pandas as pd
import re, os, time

drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/'
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Model-based topic labelling

Each question is labelled with the single primary condition it is mainly about, or
"Other" when it falls outside the ten target conditions. This replaces the keyword
rule, which returned whichever keyword happened to appear first.

In [13]:
from langchain_openai import ChatOpenAI

# The ten target primary-care conditions.
target_diseases = [
    "Hypertension", "Diabetes", "Asthma", "COPD", "Coronary Heart Disease",
    "Heart Failure", "Chronic Back Pain", "Depression", "Osteoarthritis", "Dementia"
]

classifier = ChatOpenAI(model="openai/gpt-4o-mini", api_key=os.environ["OPENROUTER_API_KEY"],
                        base_url="https://openrouter.ai/api/v1", temperature=0, max_tokens=16, max_retries=6)

SYSTEM = ("A clinical case is labelled by which of these conditions it centrally involves — "
          "the patient's main condition, or the condition the question focuses on:\n" +
          "\n".join(f"- {d}" for d in target_diseases) +
          "\nLabel it Other only when none of these conditions is central to the case. "
          "The reply is exactly one label from the list, or the word Other, and nothing else.")


def classify_disease(text):
    reply = classifier.invoke([{"role": "system", "content": SYSTEM},
                               {"role": "user", "content": str(text)[:2000]}]).content.strip()
    for disease in target_diseases:
        if disease.lower() in reply.lower():
            return disease
    return "Other"

# Load the bilingual pool and label every question. A new column, not a new file,
# so the original data is left untouched.
df_full = pd.read_csv("GP_Top10_Bilingual_Open_EndedQ.csv")
question_col = 'English_Open_Question' if 'English_Open_Question' in df_full.columns else 'English_Question'

labels = []
for i, text in enumerate(df_full[question_col]):
    try:
        labels.append(classify_disease(text))
    except Exception as e:
        labels.append("Other"); print("error at", i, str(e)[:80]); time.sleep(5)
    if i % 25 == 0:
        print(f"{i}/{len(df_full)} labelled")
    time.sleep(0.3)

df_full['Disease'] = labels
df_full.to_csv(DRIVE_PATH + "MedQA_AI_Classified.csv", index=False, encoding="utf-8-sig")
print(df_full['Disease'].value_counts())

0/715 labelled
25/715 labelled
50/715 labelled
75/715 labelled
100/715 labelled
125/715 labelled
150/715 labelled
175/715 labelled
200/715 labelled
225/715 labelled
250/715 labelled
275/715 labelled
300/715 labelled
325/715 labelled
350/715 labelled
375/715 labelled
400/715 labelled
425/715 labelled
450/715 labelled
475/715 labelled
500/715 labelled
525/715 labelled
550/715 labelled
575/715 labelled
600/715 labelled
625/715 labelled
650/715 labelled
675/715 labelled
700/715 labelled
Disease
Other                     482
Diabetes                   47
Heart Failure              33
Coronary Heart Disease     32
Hypertension               25
Depression                 24
Dementia                   20
COPD                       20
Asthma                     20
Chronic Back Pain           8
Osteoarthritis              4
Name: count, dtype: int64


## Balanced draw to 200 questions

An equal share is taken from each condition. When a condition holds fewer than the
target, the shortfall is topped up from the remaining on-topic questions so the
final set is exactly 200 where the pool allows. The per-condition counts are
printed so the balance is visible.

In [14]:
TARGET_TOTAL = 200
per_disease = TARGET_TOTAL // len(target_diseases)   # 20 each

# Keep only the ten target conditions (drop "Other").
df_filtered = df_full[df_full['Disease'].isin(target_diseases)]

sampled = []
for disease in target_diseases:
    subset = df_filtered[df_filtered['Disease'] == disease]
    sampled.append(subset.sample(min(per_disease, len(subset)), random_state=42))
df_selected = pd.concat(sampled)

# Top up to the target from the surplus of the better-represented conditions.
shortfall = TARGET_TOTAL - len(df_selected)
if shortfall > 0:
    remaining = df_filtered.drop(index=df_selected.index)
    df_selected = pd.concat([df_selected, remaining.sample(min(shortfall, len(remaining)), random_state=42)])

df_200 = df_selected.sample(frac=1, random_state=42).reset_index(drop=True)
print("Final size:", len(df_200))
print(df_200['Disease'].value_counts())

Final size: 200
Disease
Diabetes                  32
Heart Failure             29
Coronary Heart Disease    25
Hypertension              21
Depression                21
Dementia                  20
COPD                      20
Asthma                    20
Chronic Back Pain          8
Osteoarthritis             4
Name: count, dtype: int64


## Ground-truth answer text

The correct option letter is expanded to its answer text for both languages, so the
evaluation can compare the generated answer against the reference.

In [15]:
def extract_answer_text(question, correct_letter):
    if pd.isna(question) or pd.isna(correct_letter):
        return ""
    pattern = rf"\({correct_letter}\)\s*(.*?)(?=\n\s*\([A-Z]\)|\n\s*\*\*Answer|\Z)"
    match = re.search(pattern, question, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip().strip('"')
    try:
        parts = question.split(f"({correct_letter})")
        if len(parts) > 1:
            return parts[1].split("\n")[0].strip().strip('"')
    except Exception:
        pass
    return ""

df_200['English_Correct_Text'] = df_200.apply(lambda r: extract_answer_text(r['English_Question'], r['Correct_Answer']), axis=1)
df_200['German_Correct_Text'] = df_200.apply(lambda r: extract_answer_text(r['German_Question'], r['Correct_Answer']), axis=1)
df_200[['Disease', 'Correct_Answer', 'English_Correct_Text', 'German_Correct_Text']].head()

,Disease,Correct_Answer,English_Correct_Text,German_Correct_Text
0,Coronary Heart Disease,A,Myocardial perfusion scan under pharmacologica...,Myokardperfusionsszintigraphie unter pharmakol...
1,Hypertension,A,Hyperplasia of juxtaglomerular cells,Hyperplasie der juxtaglomerulären Zellen
2,Diabetes,B,Impaired iron absorption,Eingeschränkte Eisenaufnahme
3,Dementia,C,Sodium bicarbonate,Natriumbicarbonat
4,Depression,A,Glycine,Glycin


## Save the evaluation dataset

Saved under a new name so the previous golden dataset is preserved for comparison.
The retrieval notebook reads this file.

In [16]:
final_columns = ['Disease', 'English_Open_Question', 'English_Correct_Text',
                 'German_Open_Question', 'German_Correct_Text']
available = [c for c in final_columns if c in df_200.columns]
df_ragas_200 = df_200[available]

OUT = DRIVE_PATH + "AWMF_Golden_Dataset_200Q_AIFiltered.csv"
df_ragas_200.to_csv(OUT, index=False, encoding="utf-8-sig")
print("Saved:", OUT, "shape", df_ragas_200.shape)
df_ragas_200.head()

Saved: /content/drive/MyDrive/AWMF_Golden_Dataset_200Q_AIFiltered.csv shape (200, 5)


,Disease,English_Open_Question,English_Correct_Text,German_Open_Question,German_Correct_Text
0,Coronary Heart Disease,A 58-year-old woman comes to the physician bec...,Myocardial perfusion scan under pharmacologica...,Eine 58-jährige Frau kommt wegen intermittiere...,Myokardperfusionsszintigraphie unter pharmakol...
1,Hypertension,A 64-year-old man with coronary artery disease...,Hyperplasia of juxtaglomerular cells,Ein 64-jähriger Mann mit koronarer Herzkrankhe...,Hyperplasie der juxtaglomerulären Zellen
2,Diabetes,A 52-year-old woman presents to her primary ca...,Impaired iron absorption,Welche Antworten ist mit der wahrscheinlichste...,Eingeschränkte Eisenaufnahme
3,Dementia,A 81-year-old man is brought to the emergency ...,Sodium bicarbonate,Ein 81-jähriger Mann wird in die Notaufnahme g...,Natriumbicarbonat
4,Depression,A 67-year-old woman with depression comes to t...,Glycine,Eine 67-jährige Frau mit Depressionen sucht de...,Glycin
